# arcrun — Complete Walkthrough

**The async execution engine for autonomous agents.**

arcrun takes an arcllm model, a set of tools, and a task — then loops until the task is done. This notebook walks through every feature in spec 001 (Core Loop + ReAct) with **real API calls** using arcllm + Anthropic.

```
┌─────────────────────────────────┐
│  YOUR AGENT (you build this)   │
│  prompts · tools · config      │
├─────────────────────────────────┤
│  arcrun            ◄ this      │
│  loop · sandbox · events       │
├─────────────────────────────────┤
│  arcllm                        │
│  providers · transport         │
└─────────────────────────────────┘
```

**Table of Contents**

1. [Setup — Load arcllm + Bridge](#1-setup--load-arcllm--bridge)
2. [Tools — Defining What the Model Can Do](#2-tools--defining-what-the-model-can-do)
3. [run() — The Blocking Entry Point (REAL API)](#3-run--the-blocking-entry-point-real-api)
4. [LoopResult — What You Get Back](#4-loopresult--what-you-get-back)
5. [Events — The Full Audit Trail](#5-events--the-full-audit-trail)
6. [Sandbox — Permission Enforcement](#6-sandbox--permission-enforcement)
7. [Dynamic Tool Registry — Add/Remove Mid-Execution](#7-dynamic-tool-registry--addremove-mid-execution)
8. [run_async() + RunHandle — Non-Blocking Execution](#8-run_async--runhandle--non-blocking-execution)
9. [Steering — Interrupt & Follow-Up](#9-steering--interrupt--follow-up)
10. [Cancel — Hard Stop](#10-cancel--hard-stop)
11. [Context Transform — Preventing Overflow](#11-context-transform--preventing-overflow)
12. [Error Handling](#12-error-handling)
13. [Parameter Validation (jsonschema)](#13-parameter-validation-jsonschema)
14. [Max Turns — Loop Limits](#14-max-turns--loop-limits)
15. [Factory Tools — Stateful Closures (REAL API)](#15-factory-tools--stateful-closures-real-api)
16. [Full Integration Example (REAL API)](#16-full-integration-example-real-api)
17. [Architecture Summary](#17-architecture-summary)

---

## 1. Setup — Load arcllm Model

arcrun uses arcllm's types natively — `Message`, `TextBlock`, `ToolUseBlock`, `ToolResultBlock`, and `Tool` are shared between both packages. No bridge or adapter needed.

Just `load_model()` and pass it to `run()`.

In [ ]:
import sys, os

# Add arcrun src to path
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

# Load API key from .env
from dotenv import load_dotenv
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))
load_dotenv()  # also try CWD

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env"
print(f"API key loaded: {os.environ['ANTHROPIC_API_KEY'][:12]}...")

In [ ]:
from arcllm import load_model

# Load the real model — using Haiku for speed + low cost
# telemetry=True enables per-call cost tracking (auto-priced from provider metadata)
model = load_model("anthropic", "claude-haiku-4-5-20251001", telemetry=True)

print(f"Model loaded: claude-haiku-4-5 via arcllm")
print(f"Cost tracking: enabled")
print(f"No bridge needed — arcrun uses arcllm types natively")

---

## 2. Tools — Defining What the Model Can Do

A `Tool` is a dataclass with five fields:

| Field | Type | Purpose |
|-------|------|---------|
| `name` | `str` | Unique identifier (shown to model) |
| `description` | `str` | What it does (shown to model) |
| `input_schema` | `dict` | JSON Schema for parameter validation |
| `execute` | `async (params, ctx) -> str` | The actual implementation |
| `timeout_seconds` | `float \| None` | Per-tool timeout (overrides global) |

Key design decisions:
- **Async only** — matches arcllm's async-native design
- **Returns `str`** — simplest contract; errors via exceptions
- **Dataclass + factories** — no subclassing needed for stateful tools
- **JSON Schema validation** — params validated before `execute()` runs
- **Per-tool timeout** — optional, overrides the global `tool_timeout` from `run()`

In [1]:
from arcrun import Tool, ToolContext
import json


# --- Tool 1: Get current weather (simulated) ---

async def get_weather(params: dict, ctx: ToolContext) -> str:
    """Return weather data for a city."""
    city = params["city"]
    # Simulated weather data — in production, call a real API
    data = {
        "new york": {"temp_f": 42, "condition": "Cloudy", "humidity": 65},
        "san francisco": {"temp_f": 58, "condition": "Foggy", "humidity": 80},
        "miami": {"temp_f": 78, "condition": "Sunny", "humidity": 70},
    }
    weather = data.get(city.lower(), {"temp_f": 55, "condition": "Unknown", "humidity": 50})
    return json.dumps({"city": city, **weather})

weather_tool = Tool(
    name="get_weather",
    description="Get current weather for a city. Returns temperature (F), condition, and humidity.",
    input_schema={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name"},
        },
        "required": ["city"],
    },
    execute=get_weather,
)


# --- Tool 2: Calculator ---

async def calculate(params: dict, ctx: ToolContext) -> str:
    """Safely evaluate a math expression."""
    expr = params["expression"]
    # Only allow safe math operations
    allowed = set("0123456789+-*/().% ")
    if not all(c in allowed for c in expr):
        return f"Error: expression contains disallowed characters"
    try:
        result = eval(expr)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

calc_tool = Tool(
    name="calculate",
    description="Evaluate a mathematical expression. Supports +, -, *, /, (), %.",
    input_schema={
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "Math expression to evaluate"},
        },
        "required": ["expression"],
    },
    execute=calculate,
)


# --- Tool 3: Note-taker (accumulates notes) ---

notes = []

async def take_note(params: dict, ctx: ToolContext) -> str:
    """Save a note."""
    note = params["note"]
    notes.append(note)
    return f"Note saved ({len(notes)} total)"

note_tool = Tool(
    name="take_note",
    description="Save a note for later reference. Use for recording observations or findings.",
    input_schema={
        "type": "object",
        "properties": {
            "note": {"type": "string", "description": "The note to save"},
        },
        "required": ["note"],
    },
    execute=take_note,
)


print(f"Tools defined:")
for t in [weather_tool, calc_tool, note_tool]:
    print(f"  {t.name}: {t.description}")

Tools defined:
  get_weather: Get current weather for a city. Returns temperature (F), condition, and humidity.
  calculate: Evaluate a mathematical expression. Supports +, -, *, /, (), %.
  take_note: Save a note for later reference. Use for recording observations or findings.


### ToolContext — What Your Tool Receives

Every `execute()` call receives a `ToolContext` with environment info:

| Field | Type | Purpose |
|-------|------|---------|
| `run_id` | `str` | Unique ID for this execution run |
| `tool_call_id` | `str` | Correlation ID for this specific call |
| `turn_number` | `int` | Current loop iteration |
| `event_bus` | `EventBus` | Emit custom events from inside tools |
| `cancelled` | `asyncio.Event` | Check if the run was cancelled |

In [2]:
import asyncio

# ToolContext is created by arcrun and passed to your tool.
# Here's what it looks like:

example_ctx = ToolContext(
    run_id="abc-123",
    tool_call_id="tc-456",
    turn_number=1,
    event_bus=None,
    cancelled=asyncio.Event(),
)

print(f"run_id:       {example_ctx.run_id}")
print(f"tool_call_id: {example_ctx.tool_call_id}")
print(f"turn_number:  {example_ctx.turn_number}")
print(f"cancelled:    {example_ctx.cancelled.is_set()}")

run_id:       abc-123
tool_call_id: tc-456
turn_number:  1
cancelled:    False


---

## 3. run() — The Blocking Entry Point (REAL API)

The simplest way to use arcrun. Call `await run()` with a real arcllm model and get a `LoopResult` back.

This is a **live API call** — Claude will actually reason about the task, choose tools, and use them.

In [3]:
from arcrun import run

# REAL API CALL — Claude picks and uses tools autonomously
result = await run(
    model=model,
    tools=[weather_tool, calc_tool],
    system_prompt="You are a helpful weather assistant. Be concise.",
    task="What's the weather in New York and Miami? What's the temperature difference?",
    max_turns=5,
)

print(f"Answer: {result.content}")
print(f"\nTurns: {result.turns}")
print(f"Tool calls: {result.tool_calls_made}")
print(f"Tokens: {result.tokens_used}")
print(f"Cost: ${result.cost_usd:.4f}")

NameError: name 'model' is not defined

---

## 4. LoopResult — What You Get Back

`run()` returns a `LoopResult` with everything about the execution:

| Field | Type | Description |
|-------|------|-------------|
| `content` | `str \| None` | Final text response from the model |
| `turns` | `int` | Number of loop iterations |
| `tool_calls_made` | `int` | Total successful tool invocations |
| `tokens_used` | `dict` | `{"input": N, "output": N, "total": N}` |
| `strategy_used` | `str` | Which strategy ran ("react" in Phase 1) |
| `cost_usd` | `float` | Estimated total cost |
| `events` | `list[Event]` | Complete audit trail |

In [ ]:
# Inspect the full LoopResult from the previous run

print("=== LoopResult ===")
print(f"content:         {result.content[:80]}..." if result.content and len(result.content) > 80 else f"content:         {result.content}")
print(f"turns:           {result.turns}")
print(f"tool_calls_made: {result.tool_calls_made}")
print(f"tokens_used:     {result.tokens_used}")
print(f"strategy_used:   {result.strategy_used}")
print(f"cost_usd:        ${result.cost_usd:.6f}")
print(f"events count:    {len(result.events)}")

---

## 5. Events — The Full Audit Trail

**Every action emits an event. Always. Non-negotiable.** This is arcrun's core audit capability.

### All Event Types (Phase 1 + 2)

| Event | When | Data Keys |
|-------|------|----------|
| `loop.start` | `run()` called | task, tool_names, strategy |
| `loop.complete` | Finished | content, turns, tool_calls, tokens, cost |
| `loop.max_turns` | Hit turn limit | turns_used, max_turns |
| `strategy.selected` | Strategy chosen | strategy |
| `turn.start` / `turn.end` | Each iteration | turn_number |
| `llm.call` | model.invoke() | model, stop_reason, tokens, latency_ms, cost_usd |
| `tool.start` / `tool.end` | Tool execution | name, arguments / result_length, duration_ms |
| `tool.denied` | Sandbox blocked | name, reason |
| `tool.error` | Tool exception or timeout | name, error |
| `tool.registered` | New tool added to registry | name |
| `tool.replaced` | Existing tool replaced in registry | name |
| `tool.removed` | Tool removed from registry | name |

In [ ]:
# View every event from the previous REAL run

print("=== Event Timeline (from real API call) ===")
for i, event in enumerate(result.events):
    # Truncate large data values for display
    data_str = str(event.data)
    if len(data_str) > 100:
        data_str = data_str[:100] + "..."
    print(f"  {i+1:2d}. [{event.type:20s}] {data_str}")

In [ ]:
# Real-time event handling with on_event callback
# React to events AS THEY HAPPEN during a live API call

def live_logger(event):
    """Called synchronously inline as each event fires."""
    if event.type == "tool.start":
        print(f"  TOOL START: {event.data['name']}({event.data['arguments']})")
    elif event.type == "tool.end":
        print(f"  TOOL END:   {event.data['name']} -> {event.data['result_length']} chars ({event.data['duration_ms']:.0f}ms)")
    elif event.type == "llm.call":
        print(f"  LLM CALL:   stop={event.data['stop_reason']}, latency={event.data['latency_ms']:.0f}ms")
    elif event.type == "loop.complete":
        print(f"  COMPLETE:   {event.data['turns']} turns, {event.data['tool_calls']} tool calls")

print("Live event stream from real API call:")
print("-" * 50)

result = await run(
    model, [weather_tool, calc_tool],
    "You are a concise assistant.",
    "What's 15% of the temperature in San Francisco?",
    on_event=live_logger,
    max_turns=5,
)

print("-" * 50)
print(f"\nFinal answer: {result.content}")

In [ ]:
from arcrun import EventBus

# EventBus internals — you can use it standalone too
bus = EventBus(run_id="demo-run")

e1 = bus.emit("custom.start", {"message": "hello"})
e2 = bus.emit("custom.end", {"message": "done"})

print(f"Event 1: type={e1.type}, run_id={e1.run_id}, data={e1.data}")
print(f"Event 2: type={e2.type}, run_id={e2.run_id}, data={e2.data}")
print(f"Total events: {len(bus.events)}")

---

## 6. Sandbox — Permission Enforcement

The sandbox uses an **allowlist model** — only explicitly whitelisted tools can execute.

**Check order:**
1. No config -> all allowed
2. `allowed_tools` set and tool not in list -> denied
3. `check` callback provided -> delegate
4. All pass -> allowed

When denied: `tool.denied` event emits, error returned to model (it adjusts), loop continues.

In [ ]:
from arcrun import SandboxConfig

# REAL API CALL with sandbox — only weather allowed, calculate blocked
# Claude will try to calculate but get denied, then respond without it

sandbox = SandboxConfig(allowed_tools=["get_weather"])  # calculate NOT allowed

result = await run(
    model, [weather_tool, calc_tool],
    "You are a concise assistant. Use tools when helpful.",
    "What's the weather in Miami? What's 78 minus 32?",
    sandbox=sandbox,
    max_turns=5,
)

denied = [e for e in result.events if e.type == "tool.denied"]
print(f"Denied tool calls: {len(denied)}")
for d in denied:
    print(f"  Tool: {d.data['name']} — Reason: {d.data['reason']}")

print(f"\nFinal answer: {result.content}")
print(f"Turns: {result.turns}, Tool calls (successful): {result.tool_calls_made}")

In [ ]:
# Custom checker — block weather queries for specific cities

async def city_blocker(tool_name: str, params: dict) -> tuple[bool, str]:
    """Block weather queries for restricted cities."""
    blocked_cities = ["area 51", "north korea"]
    if tool_name == "get_weather":
        city = params.get("city", "").lower()
        if any(blocked in city for blocked in blocked_cities):
            return False, f"weather data for '{city}' is classified"
    return True, ""

sandbox = SandboxConfig(
    allowed_tools=["get_weather", "calculate"],
    check=city_blocker,
)

result = await run(
    model, [weather_tool, calc_tool],
    "You are a concise assistant.",
    "Get weather for New York and Area 51.",
    sandbox=sandbox,
    max_turns=5,
)

denied = [e for e in result.events if e.type == "tool.denied"]
print(f"Denied: {len(denied)}")
for d in denied:
    print(f"  {d.data['name']}: {d.data['reason']}")

print(f"\nAnswer: {result.content}")

In [ ]:
# Callback errors = fail-safe denial

async def broken_checker(tool_name: str, params: dict) -> tuple[bool, str]:
    raise RuntimeError("checker crashed!")

from arcrun.sandbox import Sandbox

bus = EventBus(run_id="test")
sandbox_obj = Sandbox(config=SandboxConfig(check=broken_checker), event_bus=bus)

allowed, reason = await sandbox_obj.check("get_weather", {"city": "NYC"})
print(f"Allowed: {allowed}")
print(f"Reason:  {reason}")
print("Crash caught — fail-safe denial. Security over availability.")

---

## 7. Dynamic Tool Registry — Add/Remove Mid-Execution

The `ToolRegistry` is mutable. Tools can be added, removed, or replaced while the loop runs. The strategy re-reads the registry **each turn**.

| Method | Behavior |
|--------|----------|
| `add(tool)` | Add new → emits `tool.registered`. Replace existing → emits `tool.replaced` |
| `remove(name)` | Remove by name. Emits `tool.removed` |
| `get(name)` | Returns `Tool` or `None` |
| `list_schemas()` | Schemas for model.invoke() |
| `names()` | List of tool names |

In [ ]:
from arcrun import ToolRegistry

bus = EventBus(run_id="registry-demo")
registry = ToolRegistry(tools=[weather_tool], event_bus=bus)

print(f"Initial: {registry.names()}")

registry.add(calc_tool)
print(f"After add: {registry.names()}")

registry.remove("get_weather")
print(f"After remove: {registry.names()}")

t = registry.get("calculate")
print(f"\nGet 'calculate': {t.name} — {t.description}")

print(f"\nRegistry events:")
for e in bus.events:
    print(f"  [{e.type}] {e.data}")

---

## 8. run_async() + RunHandle — Non-Blocking Execution

`run_async()` returns a `RunHandle` immediately while the loop runs in a background task.

```python
handle = await run_async(model, tools, prompt, task)
handle.steer()      # interrupt
handle.follow_up()  # queue for end_turn
handle.cancel()     # hard stop
handle.result()     # await LoopResult
handle.state        # read-only access
```

In [ ]:
from arcrun import run_async

# REAL API — non-blocking
handle = await run_async(
    model, [weather_tool],
    "You are a concise weather assistant.",
    "What's the weather in San Francisco?",
    max_turns=5,
)

# Access state while running
print(f"Run ID: {handle.state.run_id}")
print(f"Tools:  {handle.state.registry.names()}")

# Await completion
result = await handle.result()
print(f"\nResult: {result.content}")
print(f"Turns: {result.turns}, Cost: ${result.cost_usd:.6f}")

---

## 9. Steering — Interrupt & Follow-Up

### `steer(message)` — Interrupt
Inject between tool executions. Skips remaining tools in current response.

### `follow_up(message)` — Queue
Inject at `end_turn`. Continues loop instead of returning.

These use MockModel for deterministic behavior (real API timing makes steering demos unpredictable).

In [ ]:
# MockModel for deterministic steering demos
from dataclasses import dataclass, field

@dataclass
class MockUsage:
    input_tokens: int = 10
    output_tokens: int = 5
    total_tokens: int = 15

@dataclass
class MockToolCall:
    id: str
    name: str
    arguments: dict

@dataclass
class MockResponse:
    content: str | None = None
    tool_calls: list = field(default_factory=list)
    stop_reason: str = "end_turn"
    usage: MockUsage = field(default_factory=MockUsage)
    cost_usd: float = 0.001

class MockModel:
    def __init__(self, responses):
        self._responses = list(responses)
        self._call_count = 0
    async def invoke(self, messages, tools=None):
        if self._call_count >= len(self._responses):
            raise RuntimeError("MockModel exhausted")
        resp = self._responses[self._call_count]
        self._call_count += 1
        return resp

print("MockModel ready for steering demos.")

In [ ]:
# follow_up: inject additional work at end_turn

mock = MockModel([
    MockResponse(content="Task 1 done.", stop_reason="end_turn"),
    MockResponse(
        tool_calls=[MockToolCall(id="tc1", name="get_weather", arguments={"city": "Miami"})],
        stop_reason="tool_use",
    ),
    MockResponse(content="Task 2: Miami is 78F and sunny.", stop_reason="end_turn"),
])

handle = await run_async(mock, [weather_tool], "Be helpful.", "Do task 1")
await handle.follow_up("Also check Miami weather")

result = await handle.result()
print(f"Final: {result.content}")
print(f"Turns: {result.turns}")
print("Loop continued past first end_turn because of follow_up.")

In [ ]:
# steer: inject a redirect message that the model sees on its next turn
# With mocks, steer() is called before the loop starts, so the message
# appears as a user message at the top of the first turn.

mock = MockModel([
    # Turn 1: model sees the steer message + original task, does one tool call
    MockResponse(
        tool_calls=[MockToolCall(id="tc1", name="get_weather", arguments={"city": "New York"})],
        stop_reason="tool_use",
    ),
    # Turn 2: model responds based on the redirected context
    MockResponse(content="Redirected: New York is 42F and cloudy.", stop_reason="end_turn"),
])

handle = await run_async(mock, [weather_tool], "Be helpful.", "Check all cities")
await handle.steer("Only care about New York")

result = await handle.result()
print(f"Final: {result.content}")
print(f"Tool calls made: {result.tool_calls_made}")
print("Steer injected a user message — model saw the redirect and adapted.")

---

## 10. Cancel — Hard Stop

`cancel()` sets the `cancel_event`. Running tools see `ctx.cancelled.is_set() == True`. Strategy returns a partial `LoopResult`.

In [ ]:
mock = MockModel([
    MockResponse(
        tool_calls=[MockToolCall(id="tc1", name="get_weather", arguments={"city": "NYC"})],
        stop_reason="tool_use",
    ),
    MockResponse(content="Done.", stop_reason="end_turn"),
])

handle = await run_async(mock, [weather_tool], "Be helpful.", "Long task")
await handle.cancel()

result = await handle.result()
print(f"Content: {result.content}")
print(f"Turns:   {result.turns}")
print(f"Cancel:  {handle.state.cancel_event.is_set()}")
print("Partial result returned due to cancellation.")

---

## 11. Context Transform — Preventing Overflow

The `transform_context` hook prunes messages before each `model.invoke()` call. Prevents hitting context limits on long loops.

In [ ]:
# REAL API with context pruning

transform_log = []

def context_pruner(messages):
    """Keep system + last 6 messages."""
    transform_log.append(len(messages))
    if len(messages) <= 7:
        return messages
    return [messages[0]] + messages[-6:]

result = await run(
    model, [weather_tool, calc_tool],
    "You are a concise assistant.",
    "Check weather in New York, then San Francisco, then Miami. Summarize all three.",
    transform_context=context_pruner,
    max_turns=8,
)

print(f"Message counts at each model call: {transform_log}")
print(f"Result: {result.content}")
print(f"Turns: {result.turns}")

---

## 12. Error Handling

| Error Type | Behavior |
|-----------|----------|
| **Tool exception** | Caught, `tool.error` event, error returned to model |
| **Model API error** | Bubbles up to caller |
| **Sandbox check error** | Treated as denial (fail-safe) |

In [ ]:
# REAL API — tool that crashes, model recovers

async def exploding_tool(params: dict, ctx: ToolContext) -> str:
    raise ConnectionError("database connection refused")

db_tool = Tool(
    name="query_database",
    description="Query the user database. Returns user info.",
    input_schema={
        "type": "object",
        "properties": {"user_id": {"type": "string"}},
        "required": ["user_id"],
    },
    execute=exploding_tool,
)

result = await run(
    model, [db_tool],
    "You are a concise assistant. If a tool fails, explain the error.",
    "Look up user ID 'abc123'",
    max_turns=3,
)

errors = [e for e in result.events if e.type == "tool.error"]
print(f"Tool errors: {len(errors)}")
for err in errors:
    print(f"  {err.data['name']}: {err.data['error']}")

print(f"\nModel's response: {result.content}")
print("Error caught, sent to model, model explained the failure.")

In [ ]:
# Model API errors bubble up

class BrokenModel:
    async def invoke(self, messages, tools=None):
        raise ConnectionError("API unreachable")

try:
    await run(BrokenModel(), [weather_tool], "prompt", "task")
except ConnectionError as e:
    print(f"Caught: {e}")
    print("Model errors bubble up — arcllm handles retries, not arcrun.")

---

## 13. Parameter Validation (jsonschema)

Before every `tool.execute()`, arcrun validates arguments against `input_schema`. Failed validation returns the error to the model (it can retry).

In [ ]:
# Mock to demonstrate — model sends wrong type, validation catches it

mock = MockModel([
    MockResponse(
        tool_calls=[MockToolCall(id="tc1", name="get_weather", arguments={"city": 12345})],
        stop_reason="tool_use",
    ),
    MockResponse(
        tool_calls=[MockToolCall(id="tc2", name="get_weather", arguments={"city": "New York"})],
        stop_reason="tool_use",
    ),
    MockResponse(content="New York: 42F, Cloudy.", stop_reason="end_turn"),
])

result = await run(mock, [weather_tool], "Be helpful.", "Get weather")

print(f"Content: {result.content}")
print(f"Turns: {result.turns}")
print(f"Successful tool calls: {result.tool_calls_made}")
print("\nFirst call failed validation (int != string), second succeeded.")

---

## 14. Max Turns — Loop Limits

`max_turns` (default: 25) prevents infinite loops. Emits `loop.max_turns`, returns `content=None`.

In [ ]:
# Mock model that never stops calling tools

mock = MockModel([
    MockResponse(
        tool_calls=[MockToolCall(id=f"tc{i}", name="get_weather", arguments={"city": f"City{i}"})],
        stop_reason="tool_use",
    )
    for i in range(10)
])

result = await run(mock, [weather_tool], "Be helpful.", "Keep checking", max_turns=3)

print(f"Content: {result.content}")
print(f"Turns:   {result.turns}")

max_events = [e for e in result.events if e.type == "loop.max_turns"]
print(f"Max turns event: {max_events[0].data}")
print("Loop stopped at max_turns — no infinite loops.")

---

## 15. Factory Tools — Stateful Closures (REAL API)

For stateful tools, use **factory functions**. The closure captures state — no subclassing needed.

In [ ]:
def make_todo_tool() -> Tool:
    """Factory: creates a stateful todo list tool."""
    todos: list[str] = []

    async def execute(params: dict, ctx: ToolContext) -> str:
        action = params["action"]
        if action == "add":
            item = params.get("item", "")
            todos.append(item)
            return f"Added '{item}'. Total: {len(todos)}"
        elif action == "list":
            if not todos:
                return "Todo list is empty."
            return "\n".join(f"{i+1}. {t}" for i, t in enumerate(todos))
        elif action == "clear":
            count = len(todos)
            todos.clear()
            return f"Cleared {count} items."
        return f"Unknown action: {action}"

    return Tool(
        name="todo",
        description="Manage a todo list. Actions: add (with item), list, clear.",
        input_schema={
            "type": "object",
            "properties": {
                "action": {"type": "string", "enum": ["add", "list", "clear"]},
                "item": {"type": "string", "description": "Item to add (for 'add' action)"},
            },
            "required": ["action"],
        },
        execute=execute,
    )


# REAL API — Claude uses the stateful todo tool
todo_tool = make_todo_tool()

result = await run(
    model, [todo_tool],
    "You are a concise task manager.",
    "Add 'Buy groceries', 'Write report', and 'Call dentist' to my todo list, then show the list.",
    max_turns=8,
)

print(f"Answer: {result.content}")
print(f"\nTurns: {result.turns}, Tool calls: {result.tool_calls_made}")
print("State persisted across calls via closure — no subclassing needed.")

---

## 16. Full Integration Example (REAL API)

Everything together: real model, multiple tools, sandbox, live events, token tracking, cost.

In [ ]:
# Full end-to-end: multi-tool agent with real API calls

def event_printer(event):
    prefix = {
        "tool.start": "TOOL",
        "tool.end": "TOOL",
        "llm.call": "LLM ",
        "loop.complete": "DONE",
        "turn.start": "TURN",
    }
    label = prefix.get(event.type)
    if label:
        if event.type == "tool.start":
            print(f"  [{label}] Calling {event.data['name']}({event.data['arguments']})")
        elif event.type == "tool.end":
            print(f"  [{label}] {event.data['name']} returned {event.data['result_length']} chars")
        elif event.type == "llm.call":
            print(f"  [{label}] {event.data['stop_reason']} ({event.data['latency_ms']:.0f}ms)")
        elif event.type == "turn.start":
            print(f"  [{label}] --- Turn {event.data['turn_number']} ---")
        elif event.type == "loop.complete":
            print(f"  [{label}] {event.data['turns']} turns, {event.data['tool_calls']} tool calls")

sandbox = SandboxConfig(allowed_tools=["get_weather", "calculate", "take_note"])

notes.clear()  # reset notes

print("=" * 60)
print("LIVE EXECUTION")
print("=" * 60)

result = await run(
    model=model,
    tools=[weather_tool, calc_tool, note_tool],
    system_prompt="You are a weather analyst. Check weather, do calculations, and take notes on findings. Be concise.",
    task="Compare weather in New York, San Francisco, and Miami. Calculate the average temperature. Note which city is warmest.",
    max_turns=10,
    sandbox=sandbox,
    on_event=event_printer,
)

print("=" * 60)
print("\nEXECUTION REPORT")
print("=" * 60)
print(f"Answer:     {result.content}")
print(f"Strategy:   {result.strategy_used}")
print(f"Turns:      {result.turns}")
print(f"Tool calls: {result.tool_calls_made}")
print(f"Tokens:     {result.tokens_used}")
print(f"Cost:       ${result.cost_usd:.6f}")
print(f"Events:     {len(result.events)}")

if notes:
    print(f"\nNotes saved by agent:")
    for i, n in enumerate(notes):
        print(f"  {i+1}. {n}")

print(f"\n--- Event Type Summary ---")
from collections import Counter
type_counts = Counter(e.type for e in result.events)
for t, c in sorted(type_counts.items()):
    print(f"  {t:20s}: {c}")

---

## 17. Architecture Summary

### Module Map

```
arcrun/
├── __init__.py          # Public API surface (19 lines)
├── types.py             # Tool, ToolContext, SandboxConfig, LoopResult (53 lines)
├── events.py            # Event, EventBus (46 lines)
├── sandbox.py           # Sandbox — allowlist + custom checker (56 lines)
├── registry.py          # ToolRegistry — add/remove/get (45 lines)
├── state.py             # RunState — messages, tokens, queues (32 lines)
├── loop.py              # run(), run_async(), RunHandle (135 lines)
├── executor.py          # Shared tool execution pipeline (83 lines)
├── _messages.py         # Msg, TextBlock, ToolUseBlock helpers (27 lines)
└── strategies/
    ├── __init__.py      # Strategy selection (38 lines)
    └── react.py         # ReAct loop (135 lines)
```

**Total: ~659 lines** (under 1,000 budget)

### Key Decisions

| Decision | Choice | Why |
|----------|--------|-----|
| Tool type | Dataclass + factories | No subclassing; closures for state |
| Tool.execute | Async only | Matches arcllm |
| Tool returns | `str` | Simplest contract |
| Tool timeout | Per-tool + global default | Flexible control |
| Events | Generic Event + dict | Saves ~190 lines vs typed events |
| Event handler errors | Catch + isolate | Observer can't crash engine |
| Security | Allowlist | Deny-by-default |
| Validation | jsonschema | Every call validated |
| Steering | steer + followUp | Two timing modes |
| Dynamic tools | Denied by default | Can't bypass sandbox |
| Error truncation | 200 chars for LLM, full in events | Safety + observability |

### Tool Execution Pipeline (executor.py)

```
1. Emit tool.start
2. Sandbox check → denied? emit tool.denied, return error
3. Registry lookup → not found? return error
4. Schema validation → invalid? return error
5. Build ToolContext
6. Execute with timeout (per-tool or global)
7. Timeout? emit tool.error, return error
8. Exception? truncate for LLM, full error in tool.error event
9. Emit tool.end (with duration_ms, result_length)
10. Increment tool_calls_made, return result
```

### ReAct Loop

```
while turn_count < max_turns:
    1. Check cancel
    2. Check steer_queue -> inject if present
    3. transform_context(messages) if hook
    4. response = model.invoke(messages, tools)
    5. Accumulate tokens/cost
    6. If end_turn:
       a. Check followup_queue -> inject + continue
       b. Return LoopResult
    7. For each tool_call:
       a. Execute via executor pipeline
       b. Check steer_queue -> skip remaining
    8. Append results, increment turn
```

### arcrun vs. Your Agent

| arcrun Does | Your Agent Does |
|------------|----------------|
| Execution loop | Agent definitions |
| Tool dispatch + validation | Tool discovery |
| Event emission | Session management |
| Sandbox enforcement | Config, memory, UI |

In [ ]:
# Input validation — fails fast

try:
    await run(model, [], "prompt", "task")  # empty tools
except ValueError as e:
    print(f"Caught: {e}")

# Strategy selection
from arcrun.strategies import select_strategy, _load_strategies, STRATEGIES
if not STRATEGIES:
    _load_strategies()
print(f"Available strategies: {list(STRATEGIES.keys())}")

try:
    from arcrun.state import RunState
    bus = EventBus(run_id="t")
    reg = ToolRegistry(tools=[weather_tool], event_bus=bus)
    state = RunState(messages=[], registry=reg, event_bus=bus)
    await select_strategy(["nonexistent"], None, state)
except ValueError as e:
    print(f"Caught: {e}")

In [ ]:
# Cleanup — close the httpx client
if hasattr(model, 'close'):
    await model.close()
print("Model connection closed.")

---

## Summary

This walkthrough covered every feature in arcrun spec 001 (Core Loop + ReAct):

| # | Feature | Demo Type |
|---|---------|----------|
| 1 | **Tool definition** | Code |
| 2 | **ToolContext** | Code |
| 3 | **run()** | REAL API |
| 4 | **LoopResult** | REAL API |
| 5 | **Events + on_event** | REAL API |
| 6 | **Sandbox (allowlist + checker)** | REAL API |
| 7 | **ToolRegistry** | Code |
| 8 | **run_async() + RunHandle** | REAL API |
| 9 | **Steering (steer + follow_up)** | Mock (deterministic) |
| 10 | **Cancel** | Mock (deterministic) |
| 11 | **Context transform** | REAL API |
| 12 | **Error handling** | REAL API |
| 13 | **Parameter validation** | Mock |
| 14 | **Max turns** | Mock |
| 15 | **Factory tools** | REAL API |
| 16 | **Full integration** | REAL API |

**arcrun does one thing: execute the loop. You build the agent.**